# 第二次作业 08 聚类实验与稳定性评价

核验聚类输入，执行 HAC、PAM、DBSCAN 配置，并检查评分及划分稳定性。

原编号与新文件名的对应关系、外部依赖和执行说明见 [第二次作业导航](hw02_README.md)。

先运行本文件的**环境初始化**单元格；重启内核后也应先运行初始化。各节按原顺序排列，依赖上一步生成的变量时会在节前列出。

已有输出和执行编号是整理前的运行记录，未重新生成；新增初始化单元格没有历史输出。

[上一份 07](hw02_07_dimensionality_reduction.ipynb)

## 环境初始化

此单元格集中加载公共依赖并检查脚本位置；不执行采集、安装或模型拟合。

In [ ]:
# 标准库
import importlib
import sys
from pathlib import Path

# 第三方库
import pandas as pd
from IPython.display import display

# 仅在这里配置本机项目位置。脚本和来源数据仍保留在原 railway_analysis 项目中。
ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
EXPECTED_PYTHON = Path(r"D:\anaconda\envs\pda312\python.exe")
SCRIPTS_DIR = ROOT / "scripts"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError("请选择 pda312 内核，再从本文件的环境初始化单元格开始运行。")
if not SCRIPTS_DIR.is_dir():
    raise FileNotFoundError(f"未找到作业脚本目录：{SCRIPTS_DIR}。请核对上方 ROOT 配置。")

REQUIRED_LOCAL_MODULES = (
    "stage3_algorithms",
    "stage3_clustering",
)
missing_modules = [
    name for name in REQUIRED_LOCAL_MODULES
    if not (SCRIPTS_DIR / f"{name}.py").is_file()
]
if missing_modules:
    raise FileNotFoundError(f"作业脚本缺失：{', '.join(missing_modules)}")

# 重复运行初始化时不重复插入同一路径；业务模块仍在对应步骤导入和 reload。
scripts_path = str(SCRIPTS_DIR)
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


## 问题3—核心代码1：核验前序结果并冻结聚类实验表

In [1]:
# 功能：核验问题2数据，冻结34种距离/坐标表示与198个聚类配置，保存版本和哈希。
import stage3_algorithms
import stage3_clustering
importlib.reload(stage3_algorithms)
importlib.reload(stage3_clustering)
plan = stage3_clustering.prepare(actor="user")
for key in [
    "representations", "main_parameter_representations", "unique_cluster_runs",
    "algorithm_counts", "representative_partitions", "output_dir",
]:
    print(f"{key}: {plan[key]}")
representation_table = pd.read_csv(Path(plan["output_dir"]) / "representation_inventory.csv", encoding="utf-8-sig")
display(representation_table.groupby(["family", "node_count"]).size().rename("representation_count").to_frame())


representations: 34
main_parameter_representations: 8
unique_cluster_runs: 198
algorithm_counts: {'HAC_average': 50, 'PAM': 66, 'DBSCAN': 82}
representative_partitions: 102
output_dir: D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\analysis\user_stage3_cluster_plan_20261114_171648_630964


representation_count
family    node_count                      
MDS       33                             1
          34                             1
UMAP      33                             5
          34                             5
rail_time 33                             3
          34                             9
tSNE      33                             5
          34                             5

## 问题3—核心代码2：执行三种聚类及参数/表示对照

In [2]:
# 功能：执行198个用户实际聚类配置，逐次保存真实标签、评分、参数、中心/树/核心点信息和耗时。
import stage3_algorithms
import stage3_clustering
importlib.reload(stage3_algorithms)
importlib.reload(stage3_clustering)
clustering = stage3_clustering.run(actor="user")
for key in [
    "completed_runs", "expected_runs", "unavailable_silhouette_runs",
    "numeric_clustering_runs_complete", "verification_and_stability_summary_complete",
    "output_dir",
]:
    print(f"{key}: {clustering[key]}")


1/198 HAC_average；簇数=4；噪声=0；评分状态=computed on non-noise points only
2/198 PAM；簇数=4；噪声=0；评分状态=computed on non-noise points only
3/198 DBSCAN；簇数=3；噪声=6；评分状态=computed on non-noise points only
4/198 HAC_average；簇数=4；噪声=0；评分状态=computed on non-noise points only
5/198 PAM；簇数=4；噪声=0；评分状态=computed on non-noise points only
6/198 DBSCAN；簇数=3；噪声=6；评分状态=computed on non-noise points only
7/198 HAC_average；簇数=4；噪声=0；评分状态=computed on non-noise points only
8/198 PAM；簇数=4；噪声=0；评分状态=computed on non-noise points only
9/198 DBSCAN；簇数=3；噪声=6；评分状态=computed on non-noise points only
10/198 HAC_average；簇数=4；噪声=0；评分状态=computed on non-noise points only
11/198 PAM；簇数=4；噪声=0；评分状态=computed on non-noise points only
12/198 DBSCAN；簇数=3；噪声=6；评分状态=computed on non-noise points only
13/198 HAC_average；簇数=3；噪声=0；评分状态=computed on non-noise points only
14/198 PAM；簇数=3；噪声=0；评分状态=computed on non-noise points only
15/198 HAC_average；簇数=4；噪声=0；评分状态=computed on non-noise points only
16/198 PAM；簇数=4；噪声=0；评分状态=computed on non-noise p

## 问题3—核心代码3：复核标签与评分，汇总划分稳定性

In [3]:
# 功能：检查全部标签/参数/哈希，重算评分，核验PAM交换局部最优；生成城市归属和ARI/NMI对照表。
import stage3_algorithms
import stage3_clustering
importlib.reload(stage3_algorithms)
importlib.reload(stage3_clustering)
verified = stage3_clustering.verify_and_summarize()
CLUSTER_OUT = Path(verified["output_dir"])
quality = pd.read_csv(CLUSTER_OUT / "representative_cluster_quality.csv", encoding="utf-8-sig")
metrics = pd.read_csv(CLUSTER_OUT / "cluster_metrics.csv", encoding="utf-8-sig")
stability = pd.read_csv(CLUSTER_OUT / "partition_stability.csv", encoding="utf-8-sig")
plan_dir, _, _, _ = stage3_clustering.current_cluster_plan()
representations = pd.read_csv(plan_dir / "representation_inventory.csv", encoding="utf-8-sig")
main_ids = ["access_60_alpha_5_n34", "access_60_n33"]
with pd.option_context("display.max_columns", None, "display.width", 200, "display.precision", 5):
    display(quality.loc[quality["family"].eq("rail_time") & quality["dataset_id"].isin(main_ids),
                       [
                           "dataset_id", "algorithm", "n_clusters", "n_noise",
                           "assigned_fraction", "silhouette_input",
                           "silhouette_original_rail", "silhouette_status",
                       ]])
    embedding_ids = representations.loc[representations["main_parameter_input"].eq(True) & representations["dataset_id"].eq("access_60_n33") & ~representations["family"].eq("rail_time"), "representation_id"]
    display(quality.loc[quality["representation_id"].isin(embedding_ids),
                       [
                           "family", "algorithm", "n_clusters", "n_noise",
                           "silhouette_input", "silhouette_original_rail",
                           "calinski_harabasz", "davies_bouldin",
                       ]])
    pam_runs = metrics.loc[metrics["family"].eq("rail_time") & metrics["dataset_id"].isin(main_ids) & metrics["tags"].str.contains("PAM_seed")]
    lookup = pam_runs.set_index("run_id")["dataset_id"]
    pam_stability = stability.loc[stability["comparison"].eq("PAM_seed") & stability["run_a"].isin(lookup.index)].copy()
    pam_stability["dataset_id"] = pam_stability["run_a"].map(lookup)
    display(pam_stability.groupby("dataset_id").agg(seed_pairs=("run_a", "size"),
             mean_ARI=("ari_common_assigned", "mean"), min_ARI=("ari_common_assigned", "min"), mean_NMI=("nmi_common_assigned", "mean")))
print("本批聚类核验：", verified)


,dataset_id,algorithm,n_clusters,n_noise,assigned_fraction,silhouette_input,silhouette_original_rail,silhouette_status
12,access_60_alpha_5_n34,HAC_average,4,0,1.00000,0.60464,0.60464,computed on non-noise points only
13,access_60_alpha_5_n34,PAM,4,0,1.00000,0.33785,0.33785,computed on non-noise points only
14,access_60_alpha_5_n34,DBSCAN,3,6,0.82353,0.27710,0.27710,computed on non-noise points only
30,access_60_n33,HAC_average,4,0,1.00000,0.57122,0.57122,computed on non-noise points only
31,access_60_n33,PAM,4,0,1.00000,0.38794,0.38794,computed on non-noise points only
32,access_60_n33,DBSCAN,3,5,0.84848,0.27710,0.27710,computed on non-noise points only


,family,algorithm,n_clusters,n_noise,silhouette_input,silhouette_original_rail,calinski_harabasz,davies_bouldin
69,MDS,HAC_average,4,0,0.61184,0.57122,24.22025,0.18323
70,MDS,PAM,4,0,0.30946,0.36293,20.31122,0.73764
71,MDS,DBSCAN,1,13,NaN,NaN,NaN,NaN
75,tSNE,HAC_average,4,0,0.41716,0.23339,38.32546,0.74121
76,tSNE,PAM,4,0,0.41716,0.23339,38.32546,0.74121
77,tSNE,DBSCAN,3,0,0.23379,0.21085,11.45747,0.85577
84,UMAP,HAC_average,4,0,0.45151,0.22514,50.90995,0.73140
85,UMAP,PAM,4,0,0.44352,0.23055,49.09567,0.71989
86,UMAP,DBSCAN,3,3,0.46926,0.04484,37.54113,0.59389


,seed_pairs,mean_ARI,min_ARI,mean_NMI
dataset_id,,,,
access_60_alpha_5_n34,3,1.0,1.0,1.0
access_60_n33,3,1.0,1.0,1.0


本批聚类核验： {'output_dir': 'D:\\yangz\\大三上\\机器学习\\第二次作业\\railway_analysis\\data\\analysis\\user_stage3_clustering_20261114_171710_794272', 'verified_runs': 198, 'representative_partitions': 102, 'label_hashes_passed': True, 'node_order_passed': True, 'quality_metrics_reproduced': True, 'PAM_swap_local_optima_verified': True, 'parameter_seed_stability_pairs': 176, 'representation_comparisons': 183, 'unavailable_silhouette_runs': 26, 'noise_excluded_from_scores': True, 'nonconverged_PAM_runs': 0, 'geographic_cluster_R_figures_complete': False, 'external_interpretation_complete': False, 'report_started': False, 'note': 'No automatic winner. DBSCAN degenerate partitions retained; compare coverage and flags. ARI/NMI noise-as-label and common-assigned variants both reported.'}
